In [144]:
import pandas as pd


# Games through Sunday, 15 March 2026
df = pd.read_csv("kenpom25.csv")

df.head()

,Rk,Team,Conf,W-L,NetRtg,ORtg,ORtg Rank,DRtg,DRtg Rank,AdjT,...,Luck,Luck Rank,NetRtgSOS,NetRtgSOS Rank,ORtgSOS,ORtgSOS Rank,DRtgSOS,DRtgSOS Rank,NetRtgNCSOS,NetRtgNCSOS Rank
0,1,Duke 1,ACC,32-2,38.90,128.0,4,89.1,2,65.3,...,0.049,63,14.29,15,117.0,24,102.7,10,10.07,18
1,2,Arizona 1,B12,32-2,37.66,127.7,5,90.0,3,69.8,...,0.023,127,14.97,9,117.6,15,102.7,7,3.25,102
2,3,Michigan 1,B10,31-3,37.59,126.6,8,89.0,1,70.9,...,0.045,70,16.65,3,119.1,4,102.5,5,12.49,11
3,4,Florida 1,SEC,26-7,33.79,125.5,9,91.7,6,70.5,...,-0.036,289,16.01,5,119.4,3,103.4,26,7.96,29
4,5,Houston 2,B12,28-6,33.43,124.9,14,91.4,5,63.3,...,-0.006,203,13.58,24,116.9,25,103.3,24,0.87,156


In [145]:
# Columns (stat rankings) to drop
dropCols = ["ORtg Rank", "DRtg Rank", "AdjT Rank", "Luck Rank", "NetRtgSOS Rank", "ORtgSOS Rank", "DRtgSOS Rank", "NetRtgNCSOS Rank"]

# Remove column rankings from DataFram
df = df.drop(columns=dropCols)

df.head()

,Rk,Team,Conf,W-L,NetRtg,ORtg,DRtg,AdjT,Luck,NetRtgSOS,ORtgSOS,DRtgSOS,NetRtgNCSOS
0,1,Duke 1,ACC,32-2,38.90,128.0,89.1,65.3,0.049,14.29,117.0,102.7,10.07
1,2,Arizona 1,B12,32-2,37.66,127.7,90.0,69.8,0.023,14.97,117.6,102.7,3.25
2,3,Michigan 1,B10,31-3,37.59,126.6,89.0,70.9,0.045,16.65,119.1,102.5,12.49
3,4,Florida 1,SEC,26-7,33.79,125.5,91.7,70.5,-0.036,16.01,119.4,103.4,7.96
4,5,Houston 2,B12,28-6,33.43,124.9,91.4,63.3,-0.006,13.58,116.9,103.3,0.87


In [146]:
# Break W-L into
df[["Wins", "Losses"]] = df["W-L"].str.split("-", expand=True).astype(int)

# Filter by teams that are in the tournament (have a seed at the end of their name)
df = df[df["Team"].str[-1].str.isdigit()]

df[["Team Name", "Seed"]] = df["Team"].str.rsplit(" ", expand=True, n=1)

df["Seed"] = df["Seed"].astype(int)
df = df.sort_values("Seed")
df.head()

df = df.drop(columns=["Team"])

df.to_csv("kenpom25_CLEAN.csv", index=False)

In [147]:
# use clean csv now

# Manaually made this bracket object
bracket = {
    "East": {
        1: "Duke",
        16: "Siena",
        8: "Ohio St.",
        9: "TCU",
        5: "St. John's",
        12: "Northern Iowa",
        4: "Kansas",
        13: "Cal Baptist",
        6: "Louisville",
        11: "South Florida",
        3: "Michigan St.",
        14: "North Dakota St.",
        7: "UCLA",
        10: "UCF",
        2: "Connecticut",
        15: "Furman",
    },
    "West": {
        1: "Arizona",
        16: "LIU",   # First Four winner (vs Prairie View / Lehigh)
        8: "Villanova",
        9: "Utah St.",
        5: "Wisconsin",
        12: "High Point",
        4: "Arkansas",
        13: "Hawaii",
        6: "BYU",
        11: "N.C. State",      # First Four winner (vs Texas)
        3: "Gonzaga",
        14: "Kennesaw St.",
        7: "Miami FL",
        10: "Missouri",
        2: "Purdue",
        15: "Queens",
    },
    "South": {
        1: "Florida",
        16: "Lehigh",         # First Four winner (vs Prairie View / Lehigh)
        8: "Clemson",
        9: "Iowa",
        5: "Vanderbilt",
        12: "McNeese",
        4: "Nebraska",
        13: "Troy",
        6: "North Carolina",
        11: "VCU",
        3: "Illinois",
        14: "Penn",
        7: "Saint Mary's",
        10: "Texas A&M",
        2: "Houston",
        15: "Idaho",
    },
    "Midwest": {
        1: "Michigan",
        16: "UMBC",          # First Four winner (vs UMBC / Howard)
        8: "Georgia",
        9: "Saint Louis",
        5: "Texas Tech",
        12: "Akron",
        4: "Alabama",
        13: "Hofstra",
        6: "Tennessee",
        11: "Miami OH",           # First Four winner (vs Miami OH / SMU)
        3: "Virginia",
        14: "Wright St.",
        7: "Kentucky",
        10: "Santa Clara",
        2: "Iowa St.",
        15: "Tennessee St.",
    },
}

df = pd.read_csv("kenpom25_CLEAN.csv")

# Manually fix the names, most matched each other
bracket_teams = {team for region in bracket.values() for team in region.values()}
kenpom_teams = set(df["Team Name"])
print("Not matched:", bracket_teams - kenpom_teams)


Not matched: set()


In [ ]:
import random
## v1

ratings = dict(zip(df["Team Name"], df["NetRtg"]))                                                                                                           
                                                                                                                                                                 
def predictGame(team_a, team_b):
    diff = ratings[team_a] - ratings[team_b]
    win_prob = 1 / (1 + 10 ** (-diff / 15.0))
    if random.random() < win_prob:
        return team_a
    else:
        return team_b

In [ ]:
from collections import Counter

N = 100000
#champion_counts = Counter()

round_counts = {                                                                                                                                               
      "R32": Counter(),                                                                                                                                          
      "S16": Counter(),
      "E8": Counter(),
      "F4": Counter(),
      "Championship": Counter(),
      "Champion": Counter(),
  }

for _ in range(N):
    r32 = {}
    for region in bracket:
        r32[region] = []
        seeds = list(bracket[region].keys())
        for i in range(0, len(seeds), 2):
            winner = predictGame(bracket[region][seeds[i]], bracket[region][seeds[i + 1]])
            r32[region].append(winner)
            round_counts["R32"][winner] += 1
    
    sweet16 = {}
    for region in r32:
        sweet16[region] = []
        teams = r32[region]
        for i in range(0, len(teams), 2):
            winner = predictGame(teams[i], teams[i + 1])
            sweet16[region].append(winner)
            round_counts["S16"][winner] += 1
    
    elite8 = {}
    for region in sweet16:
        elite8[region] = []
        teams = sweet16[region]
        for i in range(0, len(teams), 2):
            winner = predictGame(teams[i], teams[i + 1])
            elite8[region].append(winner)
            round_counts["E8"][winner] += 1
    
    final4 = {}
    for region in elite8:
        final4[region] = []
        teams = elite8[region]
        for i in range(0, len(teams), 2):
            winner = predictGame(teams[i], teams[i + 1])
            final4[region].append(winner)
            round_counts["F4"][winner] += 1


    semi1 = predictGame(final4["East"][0], final4["South"][0])
    semi2 = predictGame(final4["West"][0], final4["Midwest"][0])

    round_counts["Championship"][semi1] += 1
    round_counts["Championship"][semi2] += 1

    champ = predictGame(semi1, semi2)
    round_counts["Champion"][champ] += 1

for round_name, counts in round_counts.items():
      print(f"\n--- {round_name} ---")
      for team, count in counts.most_common():
          pct = count / N * 100
          if pct >= 1.0:
              print(f"  {team}: {pct:.1f}%")


    



--- R32 ---
  Florida: 99.9%
  Arizona: 99.8%
  Duke: 99.8%
  Michigan: 99.8%
  Iowa St.: 99.5%
  Purdue: 99.3%
  Houston: 99.3%
  Illinois: 99.1%
  Connecticut: 99.0%
  Gonzaga: 98.6%
  Virginia: 97.8%
  Nebraska: 97.7%
  Michigan St.: 97.2%
  Arkansas: 95.6%
  Kansas: 94.5%
  Tennessee: 93.9%
  Alabama: 92.4%
  Vanderbilt: 91.0%
  Wisconsin: 90.9%
  St. John's: 89.7%
  Texas Tech: 87.1%
  Louisville: 79.9%
  UCLA: 73.4%
  Miami FL: 69.3%
  Ohio St.: 67.1%
  Saint Mary's: 66.3%
  BYU: 63.7%
  North Carolina: 63.5%
  Iowa: 62.1%
  Georgia: 58.2%
  Kentucky: 57.9%
  Utah St.: 53.0%
  Villanova: 47.0%
  Santa Clara: 42.1%
  Saint Louis: 41.8%
  Clemson: 37.9%
  VCU: 36.5%
  N.C. State: 36.3%
  Texas A&M: 33.7%
  TCU: 32.9%
  Missouri: 30.7%
  UCF: 26.6%
  South Florida: 20.1%
  Akron: 12.9%
  Northern Iowa: 10.3%
  High Point: 9.1%
  McNeese: 9.0%
  Hofstra: 7.6%
  Miami OH: 6.1%
  Cal Baptist: 5.5%
  Hawaii: 4.4%
  North Dakota St.: 2.8%
  Troy: 2.3%
  Wright St.: 2.2%
  Kennesaw St.: 

In [150]:
miami = 0
siena = 0

for i in range(10000):
    winner = predictGame("Miami OH", "Siena")
    if winner == "Miami OH":
        miami += 1
    else:
        siena += 1

print(miami)

8200


In [154]:
def predictBracket(team_a, team_b):
      diff = ratings[team_a] - ratings[team_b]
      win_prob = 1 / (1 + 10 ** (-diff / 15.0))
      if win_prob >= 0.5:
          print(f"  {team_a} ({win_prob*100:.1f}%) over {team_b}")
          return team_a
      else:
          print(f"  {team_b} ({(1-win_prob)*100:.1f}%) over {team_a}")
          return team_b

# R64
r32 = {}
for region in bracket:
    print(f"\n--- {region} R64 ---")
    r32[region] = []
    seeds = list(bracket[region].keys())
    for i in range(0, len(seeds), 2):
        r32[region].append(predictBracket(bracket[region][seeds[i]], bracket[region][seeds[i+1]]))

# R32
sweet16 = {}
for region in r32:
    print(f"\n--- {region} R32 ---")
    sweet16[region] = []
    for i in range(0, len(r32[region]), 2):
        sweet16[region].append(predictBracket(r32[region][i], r32[region][i+1]))

# S16
elite8 = {}
for region in sweet16:
    print(f"\n--- {region} S16 ---")
    elite8[region] = []
    for i in range(0, len(sweet16[region]), 2):
        elite8[region].append(predictBracket(sweet16[region][i], sweet16[region][i+1]))

# E8
final4 = {}
for region in elite8:
    print(f"\n--- {region} E8 ---")
    final4[region] = predictBracket(elite8[region][0], elite8[region][1])

# F4
print(f"\n--- Final Four ---")
semi1 = predictBracket(final4["East"], final4["South"])
semi2 = predictBracket(final4["West"], final4["Midwest"])

print(f"\n--- Championship ---")
champ = predictBracket(semi1, semi2)
print(f"\nChampion: {champ}")


--- East R64 ---
  Duke (99.8%) over Siena
  Ohio St. (67.1%) over TCU
  St. John's (89.7%) over Northern Iowa
  Kansas (94.4%) over Cal Baptist
  Louisville (80.0%) over South Florida
  Michigan St. (97.2%) over North Dakota St.
  UCLA (73.5%) over UCF
  Connecticut (99.0%) over Furman

--- West R64 ---
  Arizona (99.8%) over LIU
  Utah St. (53.0%) over Villanova
  Wisconsin (90.9%) over High Point
  Arkansas (95.6%) over Hawaii
  BYU (63.7%) over N.C. State
  Gonzaga (98.6%) over Kennesaw St.
  Miami FL (69.3%) over Missouri
  Purdue (99.3%) over Queens

--- South R64 ---
  Florida (99.9%) over Lehigh
  Iowa (62.0%) over Clemson
  Vanderbilt (90.9%) over McNeese
  Nebraska (97.7%) over Troy
  North Carolina (63.6%) over VCU
  Illinois (99.1%) over Penn
  Saint Mary's (66.3%) over Texas A&M
  Houston (99.3%) over Idaho

--- Midwest R64 ---
  Michigan (99.8%) over UMBC
  Georgia (58.2%) over Saint Louis
  Texas Tech (87.1%) over Akron
  Alabama (92.4%) over Hofstra
  Tennessee (93.9%)

In [ ]:
## v2 — ORtg/DRtg balance-adjusted variance
# Teams that are lopsided (all offense or all defense) are more volatile.
# Use NetRtg for expected margin, but widen the scaling factor for lopsided matchups.

ortg = dict(zip(df["Team Name"], df["ORtg"]))
drtg = dict(zip(df["Team Name"], df["DRtg"]))
luck = dict(zip(df["Team Name"], df["Luck"]))
sos = dict(zip(df["Team Name"], df["NetRtgSOS"]))

avg_ortg = df["ORtg"].mean()
avg_drtg = df["DRtg"].mean()

# "balance" = how lopsided a team is (high = volatile)
df["off_edge"] = df["ORtg"] - avg_ortg
df["def_edge"] = avg_drtg - df["DRtg"]
df["balance"] = abs(df["off_edge"] - df["def_edge"])

balance = dict(zip(df["Team Name"], df["balance"]))
avg_bal = df["balance"].mean()

def predictGame2(team_a, team_b):
    diff = ratings[team_a] - ratings[team_b]
    # Lopsided matchups -> higher scale -> closer to 50/50
    combined_bal = (balance[team_a] + balance[team_b]) / 2
    scale = 15.0 + (combined_bal - avg_bal) * 1.0
    win_prob = 1 / (1 + 10 ** (-diff / scale))
    if random.random() < win_prob:
        return team_a
    else:
        return team_b

In [ ]:
## v3 — upset-aware bracket prediction

seed_map = dict(zip(df["Team Name"], df["Seed"]))

def compute_upset_score(fav, dog):
    """Score how likely an upset is based on multiple signals."""
    rtg_gap = ratings[fav] - ratings[dog]
    win_prob = 1 / (1 + 10 ** (-rtg_gap / 15.0))
    fav_luck_val = luck[fav]
    dog_luck_val = luck[dog]
    fav_bal = balance[fav]
    sos_gap = sos[dog] - sos[fav]
    
    score = (
        (1 - win_prob) * 40 +        # close game (biggest factor)
        fav_luck_val * 20 +           # lucky favorite (regression coming)
        -dog_luck_val * 15 +          # unlucky underdog (due for a bounce)
        fav_bal * 0.5 +               # lopsided favorite (volatile)
        max(0, sos_gap) * 0.3         # battle-tested underdog
    )
    return score, win_prob

def predictBracket3(team_a, team_b, round_num=1):
    diff = ratings[team_a] - ratings[team_b]
    combined_bal = (balance[team_a] + balance[team_b]) / 2
    scale = 15.0 + (combined_bal - avg_bal) * 1.0
    win_prob = 1 / (1 + 10 ** (-diff / scale))
    
    # Determine favorite by SEED (lower seed = favorite)
    seed_a = seed_map[team_a]
    seed_b = seed_map[team_b]
    if seed_a <= seed_b:
        fav, dog = team_a, team_b
        fav_prob = win_prob if ratings[team_a] >= ratings[team_b] else 1 - win_prob
    else:
        fav, dog = team_b, team_a
        fav_prob = win_prob if ratings[team_b] >= ratings[team_a] else 1 - win_prob
    
    upset_sc, _ = compute_upset_score(fav, dog)
    dog_prob = 1 - fav_prob
    
    # Threshold increases each round (upsets harder to call in later rounds)
    threshold = 18 + (round_num - 1) * 3
    min_dog_prob = 0.30 + (round_num - 1) * 0.05
    
    if upset_sc >= threshold and dog_prob >= min_dog_prob:
        print(f"  ⚠️ UPSET: ({seed_map[dog]}) {dog} ({dog_prob*100:.1f}%) over ({seed_map[fav]}) {fav}  [upset_score={upset_sc:.1f}]")
        return dog
    else:
        print(f"  ({seed_map[fav]}) {fav} ({fav_prob*100:.1f}%) over ({seed_map[dog]}) {dog}")
        return fav

# R64
r32 = {}
for region in bracket:
    print(f"\n--- {region} R64 ---")
    r32[region] = []
    seeds = list(bracket[region].keys())
    for i in range(0, len(seeds), 2):
        r32[region].append(predictBracket3(bracket[region][seeds[i]], bracket[region][seeds[i+1]], round_num=1))

# R32
sweet16 = {}
for region in r32:
    print(f"\n--- {region} R32 ---")
    sweet16[region] = []
    for i in range(0, len(r32[region]), 2):
        sweet16[region].append(predictBracket3(r32[region][i], r32[region][i+1], round_num=2))

# S16
elite8 = {}
for region in sweet16:
    print(f"\n--- {region} S16 ---")
    elite8[region] = []
    for i in range(0, len(sweet16[region]), 2):
        elite8[region].append(predictBracket3(sweet16[region][i], sweet16[region][i+1], round_num=3))

# E8
final4 = {}
for region in elite8:
    print(f"\n--- {region} E8 ---")
    final4[region] = predictBracket3(elite8[region][0], elite8[region][1], round_num=4)

# F4
print(f"\n--- Final Four ---")
semi1 = predictBracket3(final4["East"], final4["South"], round_num=5)
semi2 = predictBracket3(final4["West"], final4["Midwest"], round_num=5)

print(f"\n--- Championship ---")
champ = predictBracket3(semi1, semi2, round_num=6)
print(f"\n🏆 Champion: {champ}")